## Descripción de los datos con YData Profiling

### Proyecto:
Pacientes con problemas de hígado - India

**Autor:** Mariana Bedoya Arismendy  

### Fecha:
2026-08-18

### Descripción:

En este notebook generamos un reporte automático de perfilado con YData Profiling sobre el dataset de pacientes con posibles problemas de hígado. El objetivo es obtener una primera vista general de los datos de forma rápida: número de registros, tipos de variables, distribuciones, valores faltantes, duplicados, correlaciones y alertas de calidad.

Este reporte automático es un **complemento** del análisis exploratorio, no lo reemplaza. La interpretación detallada de cada variable y de sus relaciones se hace de forma manual en los notebooks siguientes (02, 03 y 04); aquí aprovechamos la herramienta para detectar rápidamente dónde conviene poner la atención.

## 📚 Importar librerías

Importamos las librerías base para el análisis de datos y la herramienta de perfilado. Cada notebook del proyecto es autónomo: puede ejecutarse completo desde cero sin depender de ningún otro notebook.

In [1]:
# librerías base para ciencia de datos
from pathlib import Path

import pandas as pd
from ydata_profiling import ProfileReport

/workspaces/Pacientes_porblemas_higado_india/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_71131/123593222.py:6: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 💾 Cargar datos

Cargamos el dataset desde el archivo Parquet de la etapa intermedia, que es la versión con los tipos de datos ya corregidos. Antes de leerlo verificamos que el archivo exista: si la ruta estuviera mal, preferimos detectarlo de inmediato y no continuar con un dataset distinto al del proyecto.

La ruta se construye de forma relativa a la raíz del proyecto (subimos un nivel desde `notebooks/3-analysis`), de modo que el notebook funciona para cualquier persona que clone el repositorio.

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
DATA_PATH = DATA_DIR / "02_intermediate/Pacientes_porblemas_higado_india_type_fixed.parquet"

assert DATA_PATH.exists(), f"No se encuentra el archivo del proyecto: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH, engine="pyarrow")

print(f"Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Dataset cargado: 663 filas y 11 columnas


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.9,1
1,62,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.0,1
4,72,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.4,1


## 📊 Descripción de los datos

Antes de generar el reporte revisamos la estructura general con `info()`. Queremos confirmar que los tipos de datos son los esperados: variables numéricas para las pruebas clínicas y para la edad, y variables categóricas para el género y para el target. Esta comprobación también nos sirve para verificar que estamos leyendo el dataset correcto (663 registros, 11 columnas).

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Age                         662 non-null    Int64   
 1   Gender                      655 non-null    category
 2   Total_Bilirubin             657 non-null    Float64 
 3   Direct_Bilirubin            656 non-null    Float64 
 4   Alkaline_Phosphotase        646 non-null    Float64 
 5   Alamine_Aminotransferase    644 non-null    Float64 
 6   Aspartate_Aminotransferase  651 non-null    Float64 
 7   Total_Protiens              656 non-null    Float64 
 8   Albumin                     661 non-null    Float64 
 9   Albumin_and_Globulin_Ratio  659 non-null    Float64 
 10  Dataset                     648 non-null    category
dtypes: Float64(8), Int64(1), category(2)
memory usage: 54.1 KB


El dataset tiene 663 registros y 11 columnas: 9 variables numéricas (la edad como entero y 8 pruebas clínicas como decimales) y 2 categóricas (`Gender` y `Dataset`). Todas las columnas tienen valores nulos, aunque en proporciones pequeñas; este punto se analiza en detalle más abajo y en el notebook 02.

Para el perfilado le indicamos explícitamente a la herramienta que `Gender` y `Dataset` son categóricas. Esto es importante para el target `Dataset`, que usa los valores `1` (paciente con problemas de hígado) y `2` (paciente sin problemas) almacenados como texto: si no lo declaramos, la herramienta podría intentar tratarlo como número y producir estadísticas sin sentido, como una "media" del diagnóstico.

In [4]:
type_schema = {
    "Gender": "categorical",
    "Dataset": "categorical",
}

profile = ProfileReport(
    df,
    title="Reporte de perfilado - Pacientes con problemas de hígado (India)",
    explorative=True,
    type_schema=type_schema,
)

### 💾 Guardar el reporte en un archivo HTML

Para consultar el perfilado usamos la salida en formato HTML, que es la opción más completa de YData Profiling: incluye las métricas globales, las distribuciones de cada variable, las correlaciones y las alertas de calidad en un solo documento navegable.

Como el reporte completo pesa varios megabytes, lo guardamos como archivo HTML independiente que se puede abrir en cualquier navegador, en lugar de incrustarlo dentro del notebook. A continuación extraemos con código las métricas y alertas principales, que son las que orientan el análisis manual de los notebooks siguientes.

In [5]:
OUTPUT_DIR = DATA_DIR / "08_reporting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_path = OUTPUT_DIR / "perfilado_eda.html"
profile.to_file(report_path)
print(f"Reporte guardado en: {report_path}")
print("Se puede abrir en cualquier navegador para explorarlo de forma interactiva.")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 12.26it/s]

Reporte guardado en: /workspaces/Pacientes_porblemas_higado_india/data/08_reporting/perfilado_eda.html
Se puede abrir en cualquier navegador para explorarlo de forma interactiva.


### Métricas principales del reporte

En lugar de incrustar el reporte completo, extraemos con código las métricas clave que calculó la herramienta. Así podemos comentarlas con los valores reales del dataset y dejar constancia de los resultados en el propio notebook.

In [6]:
descripcion = profile.get_description()
tabla = descripcion.table

print(f"Registros: {tabla['n']}")
print(f"Variables: {tabla['n_var']}")
print(f"Tipos detectados: {tabla['types']}")
print(
    f"Celdas con valores faltantes: {tabla['n_cells_missing']} de {tabla['n'] * tabla['n_var']} ({tabla['p_cells_missing']:.2%})"
)
print(
    f"Variables con al menos un valor faltante: {tabla['n_vars_with_missing']} de {tabla['n_var']}"
)
print(
    f"Filas duplicadas (grupos detectados por la herramienta): {tabla['n_duplicates']} ({tabla['p_duplicates']:.2%})"
)
print(f"Tamaño aproximado en memoria: {tabla['memory_size'] / 1024:.1f} KB")

Registros: 663
Variables: 11
Tipos detectados: {'Numeric': 9, 'Categorical': 2}
Celdas con valores faltantes: 98 de 7293 (1.34%)
Variables con al menos un valor faltante: 11 de 11
Filas duplicadas (grupos detectados por la herramienta): 51 (7.69%)
Tamaño aproximado en memoria: 54.3 KB


La herramienta también genera alertas automáticas cuando detecta situaciones que conviene revisar: valores faltantes por variable, duplicados o correlaciones altas entre variables. Las listamos completas porque son el punto de partida del análisis manual.

In [ ]:
print(f"Alertas generadas por el reporte: {len(descripcion.alerts)}\n")
for alerta in descripcion.alerts:
    print("-", alerta)

Alertas generadas por el reporte: 15

- Dataset has 51 (7.7%) duplicate rows
- [Alamine_Aminotransferase] is highly overall correlated with [Aspartate_Aminotransferase]
- [Albumin] is highly overall correlated with [Albumin_and_Globulin_Ratio] and 1 other fields
- [Albumin_and_Globulin_Ratio] is highly overall correlated with [Albumin]
- [Aspartate_Aminotransferase] is highly overall correlated with [Alamine_Aminotransferase] and 2 other fields
- [Direct_Bilirubin] is highly overall correlated with [Aspartate_Aminotransferase] and 1 other fields
- [Total_Bilirubin] is highly overall correlated with [Aspartate_Aminotransferase] and 1 other fields
- [Total_Protiens] is highly overall correlated with [Albumin]
- [Gender] 8 (1.2%) missing values
- [Direct_Bilirubin] 7 (1.1%) missing values
- [Alkaline_Phosphotase] 17 (2.6%) missing values
- [Alamine_Aminotransferase] 19 (2.9%) missing values
- [Aspartate_Aminotransferase] 12 (1.8%) missing values
- [Total_Protiens] 7 (1.1%) missing values


: 

## 📊 Análisis de resultados y conclusiones

Del perfilado automático se desprende lo siguiente:

**Tamaño y tipos.** El dataset tiene 663 registros y 11 variables (9 numéricas y 2 categóricas) y ocupa alrededor de 55 KB en memoria. Es un dataset pequeño, lo que hay que tener en cuenta más adelante: un futuro modelo tendrá pocos ejemplos para aprender, sobre todo de la clase minoritaria.

**Valores faltantes.** Hay 98 celdas con valores faltantes, es decir el 1.34% del total, y las 11 variables tienen al menos un nulo. Las más afectadas son `Alamine_Aminotransferase` (19 nulos, 2.9%), `Alkaline_Phosphotase` (17, 2.6%) y, de forma especialmente relevante, el target `Dataset` (15, 2.3%): esos 15 registros no podrán usarse para entrenar un modelo supervisado porque no tienen etiqueta.

**Duplicados.** La herramienta reporta 51 grupos de filas duplicadas que reúnen 111 filas (hay grupos de 2, 3 y hasta 4 repeticiones). En términos de filas sobrantes, `pandas` cuenta 60 filas duplicadas, un 9% del dataset. Como el dataset no tiene una columna de identificación de paciente, no podemos saber si se trata del mismo paciente registrado varias veces o de pacientes distintos con resultados idénticos (posible en pruebas clínicas con valores discretos). En esta etapa de EDA no eliminamos nada; la decisión queda documentada para la etapa de preparación de datos.

**Correlaciones altas.** Las alertas señalan correlaciones fuertes entre `Alamine_Aminotransferase` y `Aspartate_Aminotransferase` (las dos transaminasas), entre `Albumin`, `Total_Protiens` y `Albumin_and_Globulin_Ratio` (las tres miden proteínas) y entre `Total_Bilirubin` y `Direct_Bilirubin`. Esto anticipa un problema de redundancia: varias variables aportan información parecida. El notebook 03 lo cuantifica con la matriz de correlación.

**Distribuciones.** El reporte muestra distribuciones muy asimétricas en las pruebas hepáticas (bilirrubinas y enzimas), con valores extremos muy alejados de la mediana. El notebook 02 analiza cada una en detalle.

En resumen: el perfilado automático confirma que el dataset es válido y consistente con el proyecto, y señala tres frentes de trabajo para el análisis manual — nulos y duplicados, redundancia entre variables y valores extremos en las pruebas clínicas.

## 💡 Propuestas e ideas

- Completar el perfilado con análisis manual: descripción general y univariable (notebook 02), bivariable (notebook 03) y multivariable con heurística (notebook 04).
- Derivar reglas de validación de datos a partir de lo observado (rangos razonables por variable, categorías válidas, control de duplicados y de nulos), documentadas en el notebook 04.
- Dejar pendiente para la etapa de preparación de datos: cómo imputar los nulos, qué hacer con los duplicados y con los valores extremos, y si conviene eliminar alguna variable redundante.
- Revisar el reporte HTML completo en el navegador cuando se quiera explorar alguna variable en concreto.

## 📖 Referencias

- Documentación de YData Profiling: <https://docs.profiling.ydata.ai/>
- Documentación de pandas: <https://pandas.pydata.org/docs/>